# Host setup: hardware, Docker, GPU passthrough, start the stack

Run from the host (not inside a container). Checks disk space, the NVIDIA driver, Docker, and GPU passthrough, then prepares the data directories and starts the stack with `docker compose up -d`.

Run the cells in order. A failing check prints what to fix before you continue.

Afterward, open the `nemo` container and run `notebooks/00_setup_and_verification.ipynb` to verify everything from inside.


In [ ]:
# Import required libraries
import subprocess
import shutil
import sys
import os
import socket
from pathlib import Path

class HardwareCheckError(Exception):
    """Raised for a failed hardware/environment check."""
    pass

print("Python environment ready.")


### Step 1: Check disk space
Checks whether enough free space is available (container images and model caches need a lot of it).


In [ ]:
def check_disk_space(required_gb=250):
    print("[1/4] Checking free disk space...")
    total, used, free = shutil.disk_usage(".")
    free_gb = free / (1024 ** 3)
    print(f"   --> Free space: {free_gb:.2f} GB")

    if free_gb < required_gb:
        raise HardwareCheckError(
            f"Not enough disk space. Needed: {required_gb} GB, available: {free_gb:.2f} GB.\n"
            f"Large language models and container images both need a lot of room."
        )
    print("   OK: disk space is sufficient.\n")

try:
    check_disk_space(required_gb=250)
except HardwareCheckError as e:
    print(f"FAILED: {e}")


### Step 2: Check the NVIDIA driver & GPU availability
Checks whether `nvidia-smi` responds on the host and how much VRAM is available.


In [ ]:
def check_nvidia_smi():
    print("[2/4] Checking the host NVIDIA driver & GPU availability...")

    # 1. Print the full nvidia-smi table (optional)
    try:
        raw_result = subprocess.run(["nvidia-smi"], capture_output=True, text=True, check=True)
        print(raw_result.stdout)
    except Exception as e:
        print(f"Could not load the full nvidia-smi table: {e}")

    # 2. VRAM check
    try:
        result = subprocess.run(
            ["nvidia-smi", "--query-gpu=index,name,memory.total,memory.free", "--format=csv,noheader,nounits"],
            capture_output=True,
            text=True,
            check=True
        )
        gpus = result.stdout.strip().split("\n")
        print(f"  --> GPUs found ({len(gpus)}):")

        total_free_vram = 0
        for gpu in gpus:
            idx, name, mem_total, mem_free = [x.strip() for x in gpu.split(",")]
            free_vram = int(mem_free) / 1024
            total_free_vram += free_vram
            print(f"   - GPU {idx}: {name} | free VRAM: {free_vram:.2f} GB / {int(mem_total)/1024:.2f} GB")

        print(f"  --> Total free VRAM across all GPUs: {total_free_vram:.2f} GB")
        if total_free_vram < 40:
            print("   WARNING: free VRAM may be tight for large LLMs.")
        print("   OK: host GPU driver is reachable.\n")

    except FileNotFoundError:
        raise HardwareCheckError("'nvidia-smi' was not found. Install the NVIDIA driver on the host.")
    except subprocess.CalledProcessError as e:
        raise HardwareCheckError(f"nvidia-smi failed: {e.stderr}")

try:
    check_nvidia_smi()
except HardwareCheckError as e:
    print(f"FAILED: {e}\nFix the driver issue before continuing.")


### Step 3: Check the Docker installation & daemon
Checks whether the Docker service is running and the current user has the needed permissions.


In [ ]:
def check_docker_installation():
    print("[3/4] Checking the Docker installation & daemon status...")
    try:
        subprocess.run(["docker", "info"], capture_output=True, text=True, check=True)
        print("   OK: Docker is installed and the daemon is running.\n")
    except FileNotFoundError:
        raise HardwareCheckError("Docker CLI not found. Is Docker installed on this system?")
    except subprocess.CalledProcessError:
        raise HardwareCheckError("The Docker daemon isn't running, or the current user lacks permission (sudo / docker group).")

try:
    check_docker_installation()
except HardwareCheckError as e:
    print(f"FAILED: {e}\nStart the Docker service, or check your user permissions.")


### Step 4: Check the NVIDIA Container Toolkit (GPU passthrough)
Tests whether a Docker container can reach the GPU.


In [ ]:
def check_nvidia_container_toolkit():
    print("[4/4] Checking the NVIDIA Container Toolkit (GPU passthrough in Docker)...")
    cmd = [
        "docker", "run", "--rm", "--gpus", "all",
        "nvidia/cuda:12.0.0-base-ubuntu22.04",
        "nvidia-smi"
    ]
    try:
        subprocess.run(cmd, capture_output=True, text=True, check=True)
        print("   OK: GPU passthrough to Docker containers works.\n")
    except subprocess.CalledProcessError as e:
        raise HardwareCheckError(
            f"Docker cannot reach the GPUs. Error:\n{e.stderr}\n"
            "Make sure the nvidia-container-toolkit is installed and Docker has been restarted."
        )

try:
    check_nvidia_container_toolkit()
except HardwareCheckError as e:
    print(f"FAILED: {e}\nInstall the NVIDIA Container Toolkit (see the README).")


### Step 5: Prepare project directories & environment
Once the checks above pass, this sets up the data folders and, optionally, the `.env` tokens.


In [ ]:
from pathlib import Path
import os
import getpass
from dotenv import load_dotenv

def setup_project_environment():
    print("=== Preparing project environment ===")
    project_root = Path(os.getenv("PROJECT_ROOT", "/data/nemo-fraud-detection-notebooks"))
    dirs = [
        project_root / "notebooks" / "01_Data_Generation" / "data",
        project_root / "notebooks" / "02_Data_Curation" / "data" / "sft",
    ]

    for path in dirs:
        if path.exists():
            print(f"Folder already exists: {path}/")
        else:
            path.mkdir(parents=True, exist_ok=True)
            print(f"Folder created: {path}/")

    print()

    # Load .env if present
    env_path = Path(".env")
    if env_path.exists():
        load_dotenv(env_path)
        print("A .env file already exists and was loaded.")

    current_hf = os.getenv("HF_TOKEN", "").strip()
    current_ngc = os.getenv("NGC_API_KEY", "").strip()

    needs_input = not current_hf or not current_ngc

    if needs_input:
        print("=== API key configuration ===")
        if not current_hf:
            # getpass so the value never gets echoed into the notebook's saved output
            hf_input = getpass.getpass("Enter your Hugging Face token (optional, press Enter to skip): ").strip()
            if hf_input:
                current_hf = hf_input
                os.environ["HF_TOKEN"] = current_hf
        else:
            print("HF_TOKEN is already set.")

        if not current_ngc:
            ngc_input = getpass.getpass("Enter your NGC API key (optional, press Enter to skip): ").strip()
            if ngc_input:
                current_ngc = ngc_input
                os.environ["NGC_API_KEY"] = current_ngc
        else:
            print("NGC_API_KEY is already set.")

        env_content = []
        if current_hf:
            env_content.append(f"HF_TOKEN={current_hf}")
        if current_ngc:
            env_content.append(f"NGC_API_KEY={current_ngc}")

        if env_content:
            env_path.write_text("\n".join(env_content) + "\n")
            print("OK: .env file updated.")
    else:
        print("All required API keys are already set in the environment.")

setup_project_environment()


### Step 6: Start the stack & show the access guide
Starts the whole stack in the background and prints the server IP and the commands to reach each service.


In [ ]:
import socket
import subprocess
from pathlib import Path
import os

def get_host_ip():
    try:
        s = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
        s.connect(("8.8.8.8", 80))
        ip = s.getsockname()[0]
        s.close()
        return ip
    except Exception:
        return "localhost"

def start_stack_and_show_guide():
    print("=== 1. Check active Docker containers ===")
    try:
        ps_res = subprocess.run(["docker", "ps"], capture_output=True, text=True, check=True)
        if ps_res.stdout.strip():
            print(ps_res.stdout)
        else:
            print("No active Docker containers found.\n")
    except Exception as e:
        print(f"Could not read Docker status: {e}\n")

    print("=== 2. Check the data directory ===")
    project_root = Path(os.getenv("PROJECT_ROOT", "/data/nemo-fraud-detection-notebooks"))
    data_dir = project_root / "notebooks" / "01_Data_Generation" / "data"
    if data_dir.exists():
        print(f"OK: the directory already exists: {data_dir}\n")
    else:
        print(f"WARNING: the directory does not exist yet: {data_dir}\n")

    print("=== 3. Start the Docker Compose stack ===")
    try:
        res = subprocess.run(["docker", "compose", "up", "-d"], capture_output=True, text=True, check=True)
        print("All containers started.\n")
    except subprocess.CalledProcessError as e:
        print(f"FAILED to start the Docker stack:\n{e.stderr}\n")
        return

    server_ip = get_host_ip()

    print("=" * 65)
    print("ACCESS GUIDE")
    print("=" * 65)

    print("\n1. NeMo container:")
    print("   - Open a shell in it:")
    print("     docker exec -it nemo-24.09 bash")
    print("   - Start Jupyter inside it (if needed):")
    print("     jupyter notebook --ip=0.0.0.0 --port=8889 --no-browser --allow-root")
    print("   - Verify: open the URL in a browser. The Jupyter UI should load.")

    print("\n2. NeMo Automodel container:")
    print("   - Open a shell in it:")
    print("     docker exec -it nemo-automodel bash")
    print("   - Start Jupyter inside it (if needed):")
    print("     jupyter notebook --ip=0.0.0.0 --port=8887 --no-browser --allow-root")
    print(f"   - Web address: http://{server_ip}:8887 (or http://localhost:8887)")
    print("   - Verify: open the URL in a browser. The Jupyter UI should load.")

    print("\n3. Grafana dashboard (monitoring):")
    print(f"   - Web address: http://{server_ip}:3000")
    print("   - Default login: admin / admin")
    print("   - Verify: check that the NVIDIA DCGM exporter dashboard shows data (GPU usage/VRAM).")

    print("\n=== 4. Current Docker containers on the system ===")
    try:
        # Shows all containers, including stopped ones, so Prometheus, dcgm-exporter etc. are visible
        res = subprocess.run(["docker", "ps", "-a"], capture_output=True, text=True, check=True)
        if res.stdout.strip():
            print(res.stdout)
        else:
            print("No Docker containers found.")
    except Exception as e:
        print(f"Could not read Docker status: {e}")

start_stack_and_show_guide()


In [ ]:
import socket
import urllib.request
import json

def check_container_connectivity():
    print("\n=== 5. Network & connectivity checks ===")

    # Checked from the host, over the compose-mapped ports:
    services_to_check = [
        ("NeMo Automodel (Jupyter)", "http://localhost:8887"),
        ("NIM Llama 3 LLM", "http://localhost:8800/v1/models"),  # NIM's OpenAI-compatible endpoint
        ("Prometheus Monitoring", "http://localhost:9090"),
        ("Grafana Dashboard", "http://localhost:3000"),
        ("DCGM Exporter (GPU)", "http://localhost:9400/metrics")
    ]

    for name, url in services_to_check:
        try:
            req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
            with urllib.request.urlopen(req, timeout=3) as response:
                print(f"OK: {name} is reachable (status: {response.getcode()})")
        except Exception as e:
            print(f"WARNING: {name} not responding yet (it may still be starting): {e}")

    print("\n--- Testing the LLM connection (NIM) ---")
    try:
        url = "http://localhost:8800/v1/models"
        req = urllib.request.Request(url)
        with urllib.request.urlopen(req, timeout=5) as response:
            data = json.loads(response.read().decode())
            print(f"Connected. Models available: {[m['id'] for m in data.get('data', [])]}")
    except Exception:
        print("INFO: the NIM container is still loading the model (can take a few minutes on first start).")

check_container_connectivity()
